# 2. Open-box explainability: one decision, fully traced

Every page choice an EDP makes factors into readable curves. This notebook reproduces the trace behind Figure 9 of the paper for a single session: 14 raw signals → 7 latent problem scores → a slot-1 score that decomposes additively across candidate widgets.

In [ ]:
import os, sys
# notebooks live in notebooks/; make the repo root importable
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from edp.env import PageCompositionEnv
from edp.policies.edp import make_problem_shapes, make_modules, score_problems
from edp.config import PROBLEMS, N_SLOTS

## Pick a size-anxious session

We scan the stream for a `size_anxious_new` session so the trace has a clear dominant problem. (The env hides the persona at serving time; here we read the stream directly only to choose an illustrative example.)

In [ ]:
env = PageCompositionEnv(n=200, seed=42, source='parametric')
env.reset()
# env._stream is (persona, category, feat) — used here only to pick an example
persona, category, feat = next(s for s in env._stream if s[0] == 'size_anxious_new')
print('persona  :', persona)
print('category :', category)

## Step 1 — raw signals

The size-anxious persona concentrates mass on `size_chart`, low `size_conf`, and `return_hist`.

In [ ]:
sig_names = ['size_conf','price_sens','return_hist','style_stretch','new',
             'mobile','size_chart','tab_switch','zoom','price_dwell',
             'cart_osc','wishlist','return_view','revisit']
vals = [feat.get(s, 0.0) for s in sig_names]
plt.figure(figsize=(10, 3))
plt.bar(sig_names, vals, color='#4C72B0'); plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right'); plt.ylabel('signal value')
plt.title('Step 1: 14 raw behavioural signals'); plt.tight_layout(); plt.show()

## Step 2 — Layer-1 problem fingerprint

`score_problems` runs the PWL shape functions and produces the 7-d problem fingerprint. F3.2 (size anxiety) dominates.

In [ ]:
shapes = make_problem_shapes()
problems = score_problems(feat, shapes)
for p, v in sorted(problems.items(), key=lambda kv: -kv[1]):
    print(f'{p:5s} {v:.3f}')

## Step 3 — slot-1 score decomposition

For slot 1, each widget score is `base + Σ on_rem·remaining + Σ on_cov·coverage − slot_decay·0`. The winner wins on its on-remaining slope against F3.2, not on a high base.

In [ ]:
modules = make_modules()
remaining = dict(problems); coverage = {p: 0.0 for p in PROBLEMS}
rows = []
for name, mod in modules.items():
    base = mod['base']
    rem  = sum(w * remaining.get(p, 0.0) for p, w in mod.get('on_rem', {}).items())
    cov  = sum(w * coverage.get(p, 0.0) for p, w in mod.get('on_cov', {}).items())
    rows.append((name, base, rem, cov, base + rem + cov))
rows.sort(key=lambda r: -r[-1])
top = rows[:5]
print(f'{"widget":22s} {"base":>6s} {"on_rem":>7s} {"total":>7s}')
for name, base, rem, cov, tot in top:
    print(f'{name:22s} {base:6.2f} {rem:7.2f} {tot:7.2f}')

In [ ]:
names = [r[0] for r in top]
bases = np.array([r[1] for r in top]); rems = np.array([r[2] for r in top])
covs  = np.array([r[3] for r in top]); tots = np.array([r[4] for r in top])
x = np.arange(len(names))
plt.figure(figsize=(9, 4))
plt.bar(x, bases, label='base', color='#7F7F7F')
plt.bar(x, rems, bottom=bases, label='on_remaining', color='#C44E52')
plt.bar(x, covs, bottom=bases+rems, label='on_coverage', color='#55A868')
plt.scatter(x, tots, color='black', zorder=5, label='total')
plt.xticks(x, names, rotation=15); plt.ylabel('score contribution')
plt.title(f'Step 3: slot-1 decomposition (winner = {names[0]})')
plt.legend(); plt.tight_layout(); plt.show()

The whole decision is 14 + 7 + (22 scored widgets) named numbers. No SHAP, no surrogate model — the trace *is* the policy.